<div style="text-align: center;">
  <img src="./imagens/logo_novaims.png" alt="Logo" style="width: 150px; height: auto; margin-bottom: 10px;">
  <h1 style="margin: 0;"><strong>Machine Learning Project: Amazing International Airlines Inc.</strong></h1>
  <h2 style="margin: 0;"><strong>Part 1/2: Exploratory Data Analysis</strong></h2>
</div>

<div style="text-align: left; margin-top: 15px;">
  <p style="margin: 0;"><strong>Group 51:</strong></p>
  <ul style="margin: 0; padding-left: 20px;">
    <li>André Ferreira | 20250398</li>
    <li>Fausto Gomes | 20221915</li>
    <li>Maria Francisca Gonçalves | 20221942</li>
    <li>Miguel Matos | 20221925</li>
  </ul>
</div>

---
#### <font> Table of Contents </font> <a class="anchor" id='toc'></a> 
0. [Context](#introduction)
1. [Imports](#Imports)  
2. [Exploratory Data Analysis - Customer Data](#exploratory-data-analysis)

- 2.1. [Data Understanding](#21-data-understanding)
  - 2.1.1.[Descriptive Analysis](#descriptive-analysis)
  - 2.1.2.[Visualizations](#visualizations)

- 2.2. [GeoData](#geodata)

- 2.3. [Data Cleaning](#data-cleaning)



----

# <span style="color:#0097b2">0. Context</span>
[Back to TOC](#toc)

# <span style="color:#0097b2">1. Imports</span>
[Back to TOC](#toc)

In [23]:
from utils.functions import *
from utils.CustomPipeline import CustomPipeline
from pipelines.Missing_values_pipeline import MissingValuesDealer
from pipelines.Outliers_pipeline import OutliersDealer
from pipelines.Encoding_pipeline import EncodingDealer
from pipelines.Scaling_pipeline import ScalingDealer
import warnings

# Suppress this specific warning
warnings.filterwarnings('ignore', 
                       message='X does not have valid feature names',
                       category=UserWarning,
                       module='sklearn.utils.validation')


customer_data = pd.read_csv("../data/cleaned_data/customers_data_cleaned.csv", index_col= 0)
flights_data = pd.read_csv("../data/cleaned_data/flights_data_cleaned.csv", index_col= 0)

pd.set_option("display.max_columns", None)

In [24]:
data = pd.merge(
    customer_data, 
    flights_data, 
    on='Loyalty#', 
    how='outer' # To keep all clients registered
)

# The merge might create NaNs for customers in df_clean who had zero flight history
# For clustering, we should fill these with 0
features_to_fill = [
    'TotalFlights', 
    'TotalDistance', 
    'TotalPointsAccumulated', 
    'TotalPointsRedeemed'
]
data[features_to_fill] = data[features_to_fill].fillna(0)

# If 'RecencyInMonths' is NaN (from the 'left' merge), it means they never flew. 
# We can keep the 999 we set earlier, or set it again.
data['RecencyInMonths'] = data['RecencyInMonths'].fillna(999)

In [25]:
flights_data

,Loyalty#,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day
0,100018,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954
1,100102,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210
2,100140,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991
3,100214,112.3,364601.7,36453.77,12908.6,2021-12-01,1.018397,0.000000,0.065188,0.057415,0.000000,0.000000,0.169642,0.000000,0.224220,0.135728,0.089849,0.007416,0.250542,0.102557
4,100272,186.4,429630.5,42953.25,10891.4,2021-11-01,2.003942,0.034945,0.110526,0.078136,0.041127,0.054967,0.167454,0.070783,0.179004,0.024562,0.044607,0.051625,0.142265,0.170228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16732,999902,267.1,610159.5,61006.55,10501.8,2021-10-01,3.022339,0.099376,0.089763,0.087597,0.051506,0.073032,0.148470,0.099877,0.122778,0.007860,0.081661,0.031620,0.106458,0.243927
16733,999911,0.0,0.0,0.00,0.0,NaN,999.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16734,999940,85.5,238578.9,23855.59,5620.0,2021-12-01,1.018397,0.000000,0.000000,0.114145,0.000000,0.000000,0.000000,0.000000,0.121733,0.254003,0.000000,0.308023,0.202095,0.078082
16735,999982,22.0,52654.0,5264.00,0.0,2021-11-01,2.003942,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.017857,0.524696,0.000000,0.240502,0.216945,0.020091


In [26]:
data.head(10)

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType,rejoined_program,Days_in_prog,Cancelled_program,Enrollment_year,Enrollment_month,Location_cluster,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day
0,100011,Amelia,Ross,Amelia Ross,Canada,Ontario,Toronto,43.593187,-79.444335,W9D 4Q9,female,Bachelor,Suburban,NaN,Married,Star,2017-05-01,2017-05-01,NaN,Standard,0,0.0,0,2017,5,1,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100012,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2019-02-27,2019-02-27,NaN,Standard,0,0.0,0,2019,2,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100013,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,2017-09-20,2017-09-20,NaN,Standard,0,0.0,0,2017,9,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100014,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,2020-11-28,2020-11-28,NaN,Standard,0,0.0,0,2020,11,3,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100015,Benjamin,Wilson,Benjamin Wilson,Canada,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,2020-04-09,2020-04-09,NaN,Standard,0,0.0,0,2020,4,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100016,Emma,Martin,Emma Martin,Canada,British Columbia,Dawson Creek,55.720562,-120.160090,M4A 1E4,female,Master,Suburban,NaN,Single,Star,2020-07-21,2020-07-21,NaN,Standard,0,0.0,0,2020,7,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,100017,Grace,Fortin,Grace Fortin,Canada,British Columbia,Dawson Creek,55.751178,-120.264920,E0K 5I2,male,Master,Urban,NaN,Married,Star,2017-04-11,2017-04-11,NaN,Standard,0,0.0,0,2017,4,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,100018,Mina,Smida,Mina Smida,Canada,Alberta,Edmonton,53.544388,-113.490930,T9G 1W3,female,Bachelor,Rural,82877.0,Married,Aurora,2019-08-09,NaN,7919.20,Standard,0,874.0,0,2019,8,5,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954
8,100102,Rigoberto,Palacio,Rigoberto Palacio,Canada,Ontario,Toronto,43.653225,-79.383186,M1R 4K3,male,College,Urban,0.0,Single,Nova,2016-03-09,NaN,2887.74,Standard,0,2122.0,0,2016,3,1,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210
9,100140,Nakesha,Klaass,Nakesha Klaass,Canada,British Columbia,Dawson Creek,55.759628,-120.237660,U5I 4F1,female,College,Suburban,0.0,Divorced,Nova,2019-07-30,NaN,2838.07,Standard,0,884.0,0,2019,7,5,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991


In [27]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16757 entries, 0 to 16756
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 16757 non-null  int64  
 1   First Name               16757 non-null  object 
 2   Last Name                16757 non-null  object 
 3   Customer Name            16757 non-null  object 
 4   Country                  16757 non-null  object 
 5   Province or State        16757 non-null  object 
 6   City                     16757 non-null  object 
 7   Latitude                 16757 non-null  float64
 8   Longitude                16757 non-null  float64
 9   Postal code              16757 non-null  object 
 10  Gender                   16757 non-null  object 
 11  Education                16757 non-null  object 
 12  Location Code            16757 non-null  object 
 13  Income                   16737 non-null  float64
 14  Marital Status        

### Feature Engeneering

In [28]:
data["Avg_spent_per_flight"] = np.where(data["EnrollmentDateOpening"] >= "2019-01-01",
                                        data["Customer Lifetime Value"]/(data["Flights_per_day"] * data["Days_in_prog"]),
                                        np.nan)

"Inf" means infinity, which happens when the divisor is zero, so when "Days_in_prog" or "Flights_per_day" is zero. To handle this, we will feel it with Nan for now.

In [29]:
data["Avg_spent_per_flight"] = data["Avg_spent_per_flight"].replace(np.inf, np.nan)

In [30]:
data["Avg_spent_per_flight"].describe()

count      7491.000000
mean       1960.837518
std       12811.625099
min           6.359410
25%          51.737521
50%         147.920551
75%         571.234834
max      749883.717188
Name: Avg_spent_per_flight, dtype: float64

In [31]:
data.head(10)

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType,rejoined_program,Days_in_prog,Cancelled_program,Enrollment_year,Enrollment_month,Location_cluster,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day,Avg_spent_per_flight
0,100011,Amelia,Ross,Amelia Ross,Canada,Ontario,Toronto,43.593187,-79.444335,W9D 4Q9,female,Bachelor,Suburban,NaN,Married,Star,2017-05-01,2017-05-01,NaN,Standard,0,0.0,0,2017,5,1,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100012,Ethan,Thompson,Ethan Thompson,Canada,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2019-02-27,2019-02-27,NaN,Standard,0,0.0,0,2019,2,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100013,Layla,Young,Layla Young,Canada,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,2017-09-20,2017-09-20,NaN,Standard,0,0.0,0,2017,9,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100014,Amelia,Bennett,Amelia Bennett,Canada,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,2020-11-28,2020-11-28,NaN,Standard,0,0.0,0,2020,11,3,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100015,Benjamin,Wilson,Benjamin Wilson,Canada,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,2020-04-09,2020-04-09,NaN,Standard,0,0.0,0,2020,4,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100016,Emma,Martin,Emma Martin,Canada,British Columbia,Dawson Creek,55.720562,-120.160090,M4A 1E4,female,Master,Suburban,NaN,Single,Star,2020-07-21,2020-07-21,NaN,Standard,0,0.0,0,2020,7,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,100017,Grace,Fortin,Grace Fortin,Canada,British Columbia,Dawson Creek,55.751178,-120.264920,E0K 5I2,male,Master,Urban,NaN,Married,Star,2017-04-11,2017-04-11,NaN,Standard,0,0.0,0,2017,4,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,100018,Mina,Smida,Mina Smida,Canada,Alberta,Edmonton,53.544388,-113.490930,T9G 1W3,female,Bachelor,Rural,82877.0,Married,Aurora,2019-08-09,NaN,7919.20,Standard,0,874.0,0,2019,8,5,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954,43.156382
8,100102,Rigoberto,Palacio,Rigoberto Palacio,Canada,Ontario,Toronto,43.653225,-79.383186,M1R 4K3,male,College,Urban,0.0,Single,Nova,2016-03-09,NaN,2887.74,Standard,0,2122.0,0,2016,3,1,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210,NaN
9,100140,Nakesha,Klaass,Nakesha Klaass,Canada,British Columbia,Dawson Creek,55.759628,-120.237660,U5I 4F1,female,College,Suburban,0.0,Divorced,Nova,2019-07-30,NaN,2838.07,Standard,0,884.0,0,2019,7,5,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991,16.215326


In [32]:
rs_data = data.drop(columns= ["First Name", "Last Name", "Customer Name", "Country", "Province or State", "City", 
                            "Latitude", "Longitude", "Postal code", "Loyalty#", "Gender", "Education", "Location Code", 
                            "Marital Status", "LoyaltyStatus", "EnrollmentDateOpening", "CancellationDate", 
                            "EnrollmentType", "rejoined_program", "Cancelled_program", "LastFlightDate"])

In [33]:
rs_data.head(10)

,Income,Customer Lifetime Value,Days_in_prog,Enrollment_year,Enrollment_month,Location_cluster,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day,Avg_spent_per_flight
0,NaN,NaN,0.0,2017,5,1,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,0.0,2019,2,4,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,0.0,2017,9,5,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,0.0,2020,11,3,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,0.0,2020,4,4,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,0.0,2020,7,5,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,0.0,2017,4,5,0.0,0.0,0.00,0.0,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,82877.0,7919.20,874.0,2019,8,5,229.9,530230.0,53014.30,20562.8,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954,43.156382
8,0.0,2887.74,2122.0,2016,3,1,247.7,339114.6,33903.96,18760.6,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210,NaN
9,0.0,2838.07,884.0,2019,7,5,216.8,432030.8,43192.58,4896.0,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991,16.215326


# Data Preproc Parameters

### Kmeans

In [34]:
km = KMeans(n_clusters= 6) # Just a random but realistic number of clusters

pipeline = CustomPipeline(MissingValuesDealer(),
                          OutliersDealer(),
                          ScalingDealer(),
                          km)

params = {
    "imputer__imputation_method": ["knn", "simple", "iterative"],
    "imputer__knn_neighbors": np.arange(3, 31),
    "imputer__knn_scaling_method": ["standard", "minmax", "robust"],
    "imputer__simple_strategy_num": ["mean", "median"],
    "imputer__iterative_max_iter": [5, 10, 20, 50],
    "outlier_remover__outlier_method": ["Isolation_Forest", "LOF", "z_score"],
    "outlier_remover__threshold": [2, 2.5, 3, 3.5, 4],
    "outlier_remover__contamination_IF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__estimators_IF": [50, 100, 150, 200],
    "outlier_remover__max_samples_IF": ["auto", 0.5, 0.7, 1.0],
    "outlier_remover__n_neighbors": [5, 10, 20, 30, 50, 70, 90, 100],
    "outlier_remover__contamination_LOF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__metric_LOF": ["euclidean", "manhattan", "chebyshev", "minkowski"],
    "scaler__scaler_name": ["standard", "minmax", "robust"]
}


# Use a single train-test split where both are the full dataset
cv = [(np.arange(len(rs_data)), np.arange(len(rs_data)))]

search_km = RandomizedSearchCV(
    pipeline,
    params,
    n_iter=200,
    scoring= clustering_scorer,  
    cv= cv,
    verbose= 3,
    error_score= "raise",
    n_jobs = 4)

In [35]:
search_km.fit(rs_data)

Fitting 1 folds for each of 200 candidates, totalling 200 fits
[CV 1/1] END imputer__imputation_method=simple, imputer__iterative_max_iter=10, imputer__knn_neighbors=30, imputer__knn_scaling_method=robust, imputer__simple_strategy_num=mean, outlier_remover__contamination_IF=0.15, outlier_remover__contamination_LOF=0.03, outlier_remover__estimators_IF=50, outlier_remover__max_samples_IF=auto, outlier_remover__metric_LOF=chebyshev, outlier_remover__n_neighbors=70, outlier_remover__outlier_method=z_score, outlier_remover__threshold=2.5, scaler__scaler_name=standard;, score=0.899 total time=   3.4s
[CV 1/1] END imputer__imputation_method=iterative, imputer__iterative_max_iter=50, imputer__knn_neighbors=7, imputer__knn_scaling_method=robust, imputer__simple_strategy_num=median, outlier_remover__contamination_IF=0.15, outlier_remover__contamination_LOF=0.3, outlier_remover__estimators_IF=150, outlier_remover__max_samples_IF=0.7, outlier_remover__metric_LOF=manhattan, outlier_remover__n_neigh

,estimator,CustomPipelin...alingDealer())
,param_distributions,"{'imputer__imputation_method': ['knn', 'simple', ...], 'imputer__iterative_max_iter': [5, 10, ...], 'imputer__knn_neighbors': array([ 3, 4..., 28, 29, 30]), 'imputer__knn_scaling_method': ['standard', 'minmax', ...], ...}"
,n_iter,200
,scoring,<function clu...t 0x281449760>
,n_jobs,4
,refit,True
,cv,"[(array([ 0,...16755, 16756]), ...)]"
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,'raise'


In [39]:
km_params = pd.DataFrame(search_km.cv_results_)
km_params.to_csv("../data/models_preproc_data/km_params.csv")

joblib.dump(search_km.best_estimator_, 'models_preproc/km_preproc.pkl')
km_preproc = joblib.load('models_preproc/km_preproc.pkl')

km_params.sort_values(by="rank_test_score", ascending= True).head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_scaler__scaler_name,param_outlier_remover__threshold,param_outlier_remover__outlier_method,param_outlier_remover__n_neighbors,param_outlier_remover__metric_LOF,param_outlier_remover__max_samples_IF,param_outlier_remover__estimators_IF,param_outlier_remover__contamination_LOF,param_outlier_remover__contamination_IF,param_imputer__simple_strategy_num,param_imputer__knn_scaling_method,param_imputer__knn_neighbors,param_imputer__iterative_max_iter,param_imputer__imputation_method,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
34,0.361864,0.0,3.019255,0.0,minmax,4.0,Isolation_Forest,90,manhattan,0.7,50,0.03,0.50,mean,standard,29,5,simple,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.970455,0.970455,0.0,1
93,0.177451,0.0,2.967490,0.0,standard,2.0,Isolation_Forest,10,minkowski,auto,50,0.10,0.50,mean,standard,8,20,simple,"{'scaler__scaler_name': 'standard', 'outlier_r...",0.958659,0.958659,0.0,2
9,7.671202,0.0,9.688294,0.0,standard,4.0,LOF,5,minkowski,1.0,50,0.50,0.50,median,minmax,10,10,knn,"{'scaler__scaler_name': 'standard', 'outlier_r...",0.958452,0.958452,0.0,3
194,7.661502,0.0,9.116570,0.0,minmax,4.0,Isolation_Forest,10,euclidean,1.0,50,0.30,0.25,median,minmax,26,10,knn,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.951660,0.951660,0.0,4
127,8.233125,0.0,13.285826,0.0,minmax,4.0,LOF,5,euclidean,0.7,200,0.03,0.01,median,minmax,12,10,knn,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.949511,0.949511,0.0,5


# Hierarchical Clustering

In [41]:
hierar = AgglomerativeClustering(n_clusters = 6) # Just a random but realistic number of clusters

pipeline = CustomPipeline(MissingValuesDealer(),
                          OutliersDealer(),
                          ScalingDealer(),
                          hierar)

params = {
    "imputer__imputation_method": ["knn", "simple", "iterative"],
    "imputer__knn_neighbors": np.arange(3, 31),
    "imputer__knn_scaling_method": ["standard", "minmax", "robust"],
    "imputer__simple_strategy_num": ["mean", "median"],
    "imputer__iterative_max_iter": [5, 10, 20, 50],
    "outlier_remover__outlier_method": ["Isolation_Forest", "LOF", "z_score"],
    "outlier_remover__threshold": [2, 2.5, 3, 3.5, 4],
    "outlier_remover__contamination_IF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__estimators_IF": [50, 100, 150, 200],
    "outlier_remover__max_samples_IF": ["auto", 0.5, 0.7, 1.0],
    "outlier_remover__n_neighbors": [5, 10, 20, 30, 50, 70, 90, 100],
    "outlier_remover__contamination_LOF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__metric_LOF": ["euclidean", "manhattan", "chebyshev", "minkowski"],
    "scaler__scaler_name": ["standard", "minmax", "robust"]
}


# Use a single train-test split where both are the full dataset
cv = [(np.arange(len(rs_data)), np.arange(len(rs_data)))]

search_hierar = RandomizedSearchCV(
    pipeline,
    params,
    n_iter=200,
    scoring= clustering_scorer,  
    cv= cv,
    verbose= 3,
    error_score= "raise",
    n_jobs = 4)

In [42]:
search_hierar.fit(rs_data)

Fitting 1 folds for each of 200 candidates, totalling 200 fits
[CV 1/1] END imputer__imputation_method=simple, imputer__iterative_max_iter=50, imputer__knn_neighbors=22, imputer__knn_scaling_method=standard, imputer__simple_strategy_num=median, outlier_remover__contamination_IF=0.1, outlier_remover__contamination_LOF=0.03, outlier_remover__estimators_IF=50, outlier_remover__max_samples_IF=auto, outlier_remover__metric_LOF=minkowski, outlier_remover__n_neighbors=5, outlier_remover__outlier_method=Isolation_Forest, outlier_remover__threshold=4, scaler__scaler_name=robust;, score=0.889 total time=  19.1s
[CV 1/1] END imputer__imputation_method=iterative, imputer__iterative_max_iter=10, imputer__knn_neighbors=19, imputer__knn_scaling_method=robust, imputer__simple_strategy_num=mean, outlier_remover__contamination_IF=0.5, outlier_remover__contamination_LOF=0.03, outlier_remover__estimators_IF=50, outlier_remover__max_samples_IF=0.5, outlier_remover__metric_LOF=manhattan, outlier_remover__n_

,estimator,CustomPipelin...alingDealer())
,param_distributions,"{'imputer__imputation_method': ['knn', 'simple', ...], 'imputer__iterative_max_iter': [5, 10, ...], 'imputer__knn_neighbors': array([ 3, 4..., 28, 29, 30]), 'imputer__knn_scaling_method': ['standard', 'minmax', ...], ...}"
,n_iter,200
,scoring,<function clu...t 0x281449760>
,n_jobs,4
,refit,True
,cv,"[(array([ 0,...16755, 16756]), ...)]"
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,'raise'


In [44]:
hierar_params = pd.DataFrame(search_hierar.cv_results_)
hierar_params.to_csv("../data/models_preproc_data/hierar_params.csv")

joblib.dump(search_hierar.best_estimator_, 'models_preproc/hierar_preproc.pkl')
hierar_preproc = joblib.load('models_preproc/hierar_preproc.pkl')

hierar_params.sort_values(by="rank_test_score", ascending= True).head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_scaler__scaler_name,param_outlier_remover__threshold,param_outlier_remover__outlier_method,param_outlier_remover__n_neighbors,param_outlier_remover__metric_LOF,param_outlier_remover__max_samples_IF,param_outlier_remover__estimators_IF,param_outlier_remover__contamination_LOF,param_outlier_remover__contamination_IF,param_imputer__simple_strategy_num,param_imputer__knn_scaling_method,param_imputer__knn_neighbors,param_imputer__iterative_max_iter,param_imputer__imputation_method,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
169,4.936974,0.0,7.465002,0.0,minmax,3.5,Isolation_Forest,50,chebyshev,auto,50,0.15,0.50,mean,minmax,18,5,simple,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.961547,0.961547,0.0,1
36,7.804138,0.0,11.887815,0.0,standard,3.5,LOF,30,minkowski,0.5,150,0.01,0.25,median,minmax,20,5,simple,"{'scaler__scaler_name': 'standard', 'outlier_r...",0.955941,0.955941,0.0,2
62,12.919422,0.0,14.375455,0.0,minmax,2.0,Isolation_Forest,10,manhattan,0.7,100,0.50,0.30,median,standard,9,20,knn,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.950900,0.950900,0.0,3
143,6.537176,0.0,8.433753,0.0,minmax,3.5,Isolation_Forest,5,euclidean,0.5,200,0.05,0.30,mean,standard,11,50,simple,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.948176,0.948176,0.0,4
91,6.963298,0.0,8.499386,0.0,minmax,2.5,LOF,10,minkowski,1.0,100,0.50,0.01,median,minmax,29,20,simple,"{'scaler__scaler_name': 'minmax', 'outlier_rem...",0.947973,0.947973,0.0,5


### DBSCAN

In [45]:
db = DBSCAN(
    eps=0.5,           
    min_samples=50,    
    metric='euclidean')

pipeline = CustomPipeline(MissingValuesDealer(),
                          OutliersDealer(),
                          ScalingDealer(),
                          db)

params = {
    "imputer__imputation_method": ["knn", "simple", "iterative"],
    "imputer__knn_neighbors": np.arange(3, 31),
    "imputer__knn_scaling_method": ["standard", "minmax", "robust"],
    "imputer__simple_strategy_num": ["mean", "median"],
    "imputer__iterative_max_iter": [5, 10, 20, 50],
    "outlier_remover__outlier_method": ["Isolation_Forest", "LOF", "z_score"],
    "outlier_remover__threshold": [2, 2.5, 3, 3.5, 4],
    "outlier_remover__contamination_IF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__estimators_IF": [50, 100, 150, 200],
    "outlier_remover__max_samples_IF": ["auto", 0.5, 0.7, 1.0],
    "outlier_remover__n_neighbors": [5, 10, 20, 30, 50, 70, 90, 100],
    "outlier_remover__contamination_LOF": [0.01, 0.03, 0.05, 0.10, 0.15, 0.25, 0.3, 0.5],
    "outlier_remover__metric_LOF": ["euclidean", "manhattan", "chebyshev", "minkowski"],
    "scaler__scaler_name": ["standard", "minmax", "robust"]
}


# Use a single train-test split where both are the full dataset
cv = [(np.arange(len(rs_data)), np.arange(len(rs_data)))]

search_db = RandomizedSearchCV(
    pipeline,
    params,
    n_iter=200,
    scoring= clustering_scorer,  
    cv= cv,
    verbose= 3,
    error_score= "raise",
    n_jobs = 4)

In [48]:
search_db.fit(rs_data)

Fitting 1 folds for each of 200 candidates, totalling 200 fits
[CV 1/1] END imputer__imputation_method=simple, imputer__iterative_max_iter=20, imputer__knn_neighbors=30, imputer__knn_scaling_method=minmax, imputer__simple_strategy_num=mean, outlier_remover__contamination_IF=0.15, outlier_remover__contamination_LOF=0.1, outlier_remover__estimators_IF=50, outlier_remover__max_samples_IF=0.5, outlier_remover__metric_LOF=minkowski, outlier_remover__n_neighbors=100, outlier_remover__outlier_method=z_score, outlier_remover__threshold=2, scaler__scaler_name=minmax;, score=-1.000 total time=   1.1s
[CV 1/1] END imputer__imputation_method=iterative, imputer__iterative_max_iter=10, imputer__knn_neighbors=14, imputer__knn_scaling_method=robust, imputer__simple_strategy_num=median, outlier_remover__contamination_IF=0.15, outlier_remover__contamination_LOF=0.15, outlier_remover__estimators_IF=200, outlier_remover__max_samples_IF=1.0, outlier_remover__metric_LOF=manhattan, outlier_remover__n_neighbo

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 1/1] END imputer__imputation_method=iterative, imputer__iterative_max_iter=20, imputer__knn_neighbors=18, imputer__knn_scaling_method=standard, imputer__simple_strategy_num=mean, outlier_remover__contamination_IF=0.5, outlier_remover__contamination_LOF=0.05, outlier_remover__estimators_IF=150, outlier_remover__max_samples_IF=auto, outlier_remover__metric_LOF=minkowski, outlier_remover__n_neighbors=70, outlier_remover__outlier_method=z_score, outlier_remover__threshold=2.5, scaler__scaler_name=minmax;, score=-1.000 total time=   4.3s
[CV 1/1] END imputer__imputation_method=iterative, imputer__iterative_max_iter=50, imputer__knn_neighbors=28, imputer__knn_scaling_method=standard, imputer__simple_strategy_num=mean, outlier_remover__contamination_IF=0.05, outlier_remover__contamination_LOF=0.1, outlier_remover__estimators_IF=200, outlier_remover__max_samples_IF=0.5, outlier_remover__metric_LOF=minkowski, outlier_remover__n_neighbors=50, outlier_remover__outlier_method=Isolation_Forest,

,estimator,CustomPipelin...alingDealer())
,param_distributions,"{'imputer__imputation_method': ['knn', 'simple', ...], 'imputer__iterative_max_iter': [5, 10, ...], 'imputer__knn_neighbors': array([ 3, 4..., 28, 29, 30]), 'imputer__knn_scaling_method': ['standard', 'minmax', ...], ...}"
,n_iter,200
,scoring,<function clu...t 0x281449760>
,n_jobs,4
,refit,True
,cv,"[(array([ 0,...16755, 16756]), ...)]"
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,'raise'


In [50]:
db_params_2 = pd.DataFrame(search_db.cv_results_)
db_params_2.to_csv("../data/models_preproc_data/db_params_2.csv")

joblib.dump(search_db.best_estimator_, 'models_preproc/db_preproc_2.pkl')
db_preproc_2 = joblib.load('models_preproc/db_preproc_2.pkl')

db_params_2.sort_values(by="rank_test_score", ascending= True).head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_scaler__scaler_name,param_outlier_remover__threshold,param_outlier_remover__outlier_method,param_outlier_remover__n_neighbors,param_outlier_remover__metric_LOF,param_outlier_remover__max_samples_IF,param_outlier_remover__estimators_IF,param_outlier_remover__contamination_LOF,param_outlier_remover__contamination_IF,param_imputer__simple_strategy_num,param_imputer__knn_scaling_method,param_imputer__knn_neighbors,param_imputer__iterative_max_iter,param_imputer__imputation_method,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
197,7.631931,0.0,7.418725,0.0,standard,3.5,Isolation_Forest,5,chebyshev,auto,100,0.05,0.5,median,minmax,4,20,knn,"{'scaler__scaler_name': 'standard', 'outlier_r...",0.207777,0.207777,0.0,1
5,9.646527,0.0,12.228397,0.0,robust,4.0,Isolation_Forest,5,minkowski,1.0,200,0.15,0.5,mean,standard,3,5,knn,"{'scaler__scaler_name': 'robust', 'outlier_rem...",0.157372,0.157372,0.0,2
33,8.922321,0.0,11.969871,0.0,standard,2.5,Isolation_Forest,90,manhattan,1.0,200,0.15,0.5,mean,minmax,9,10,knn,"{'scaler__scaler_name': 'standard', 'outlier_r...",0.130309,0.130309,0.0,3
63,3.992239,0.0,5.890433,0.0,robust,2.0,Isolation_Forest,70,minkowski,0.7,150,0.03,0.5,median,robust,20,50,iterative,"{'scaler__scaler_name': 'robust', 'outlier_rem...",0.126750,0.126750,0.0,4
123,7.873188,0.0,10.028026,0.0,robust,2.5,Isolation_Forest,90,euclidean,auto,50,0.01,0.5,mean,robust,28,10,knn,"{'scaler__scaler_name': 'robust', 'outlier_rem...",0.095004,0.095004,0.0,5
